# Phase 2A: SwoR Acceptance Vector Measurement

**Goal**: Measure the per-position acceptance vector `p = [p_1, p_2, ..., p_k]` for the Llama-3.2 (1B draft, 3B target) pair, where `p_b` is the probability that the *b*-th child of a node is accepted under Sequoia-style sampling-without-replacement (SwoR) verification.

This vector is the input to Sequoia's DP tree-construction algorithm (Section 3.1 of the paper). Without measuring it, we'd be plugging estimated values into the DP — measuring it makes the Phase 2B tree predictions defensible.

**Method** (greedy-equivalent variant, matching how we'll run Phase 2B):
1. At each decoding position, draft model produces top-k candidates (ranked by probability).
2. Target model's argmax gives the "correct" next token.
3. Walk candidates in order. The first candidate matching target's argmax is "accepted at position b" where b is its rank (1-indexed). If no candidate matches, this is a full rejection.
4. Aggregate over many prompt positions: `p_b = P(target argmax == draft's b-th candidate | first b-1 didn't match)`.

This matches the *cover property* behavior of SwoR in the greedy limit: if the target's top-1 is anywhere in the draft's top-k, acceptance happens at the matching rank.

In [1]:
!pip install -q transformers accelerate

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import torch
import numpy as np
import json
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

Torch: 2.10.0+cu128, CUDA: True
GPU: Tesla T4


In [4]:
# ============================================================
# Load models (same pair as Phase 1)
# ============================================================
TARGET_MODEL = "meta-llama/Llama-3.2-3B"
DRAFT_MODEL  = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.float16, device_map="cuda"
)
target_model.eval()

draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL, torch_dtype=torch.float16, device_map="cuda"
)
draft_model.eval()

print(f"Target: {TARGET_MODEL} ({sum(p.numel() for p in target_model.parameters())/1e9:.2f}B params)")
print(f"Draft : {DRAFT_MODEL} ({sum(p.numel() for p in draft_model.parameters())/1e9:.2f}B params)")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Target: meta-llama/Llama-3.2-3B (3.21B params)
Draft : meta-llama/Llama-3.2-1B (1.24B params)


In [5]:
# ============================================================
# Prompts (same 10 as Phase 1 — consistency across phases)
# ============================================================
PROMPTS = [
    "The history of artificial intelligence began in the 1950s when researchers first",
    "In computer science, a hash table is a data structure that implements an associative",
    "The process of photosynthesis converts light energy into chemical energy through",
    'def fibonacci(n):\n    """Calculate the nth Fibonacci number."""\n    if n <= 1:\n        return n\n',
    "# Python implementation of binary search\ndef binary_search(arr, target):\n",
    "Summarize the following: Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. Machine learning algorithms use historical data as input to predict new output values. Machine learning is",
    "Write a detailed explanation of how TCP/IP networking works, starting from",
    "Explain the difference between a stack and a queue data structure. A stack",
    "The transformer architecture was introduced in the paper Attention is All You Need by Vaswani et al. in 2017. It replaced recurrent neural networks with self-attention mechanisms, enabling parallel processing of sequences. The key innovation was the multi-head attention mechanism, which allows the model to attend to different parts of the input simultaneously. Since then, transformers have become the foundation for",
    "Large language models are trained on massive datasets of text from the internet. During training, the model learns to predict the next token in a sequence given all previous tokens. This autoregressive training objective means that at inference time, the model generates text one token at a time, each conditioned on all previously generated tokens. This sequential nature creates a fundamental bottleneck because",
]

PROMPT_LABELS = [
    "general_1", "general_2", "general_3",
    "code_1", "code_2", "summarization",
    "instruction_1", "instruction_2",
    "long_ctx_1", "long_ctx_2",
]

K_MAX = 5  # measure p[1..5]
POSITIONS_PER_PROMPT = 32  # number of decoding positions we measure per prompt
print(f"Will measure p[1..{K_MAX}] over {len(PROMPTS)} prompts × {POSITIONS_PER_PROMPT} positions = {len(PROMPTS)*POSITIONS_PER_PROMPT} samples")

Will measure p[1..5] over 10 prompts × 32 positions = 320 samples


In [6]:
# ============================================================
# Measurement loop
# ============================================================
#
# For each prompt, we step forward one token at a time (using target's argmax
# to build the context — this matches greedy decoding so our vector is valid
# for the Phase 2B greedy tree). At each step we also run draft model and
# record where target's argmax appears in draft's top-k candidate list.
#
# Let match_rank[i] in {1, 2, 3, ..., K_MAX, None} be target's argmax rank
# within draft's top-K candidates at position i. Then:
#   p_1 = P(match_rank == 1)
#   p_2 = P(match_rank == 2 | match_rank > 1) — marginal at rank 2
#   p_3 = P(match_rank == 3 | match_rank > 2), etc.
#
# This is the "per-position conditional acceptance probability" that Sequoia's
# DP uses (positional acceptance assumption, Def 3.1).

@torch.inference_mode()
def measure_match_ranks(prompt, n_positions=POSITIONS_PER_PROMPT, k_max=K_MAX):
    """Walk forward n_positions tokens using target's greedy decode; at each
    step record where target's argmax falls in draft's top-k.
    Returns list of match ranks (1-indexed) or None if no match in top-k.
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    match_ranks = []

    for _ in range(n_positions):
        # Target argmax — this is what we want draft to match
        t_out = target_model(input_ids)
        target_argmax = t_out.logits[:, -1, :].argmax(dim=-1).item()

        # Draft top-k candidates at this same position
        d_out = draft_model(input_ids)
        draft_topk = torch.topk(d_out.logits[:, -1, :], k=k_max, dim=-1).indices[0].tolist()

        # Find rank of target's argmax in draft's top-k (1-indexed), or None
        try:
            rank = draft_topk.index(target_argmax) + 1
        except ValueError:
            rank = None
        match_ranks.append(rank)

        # Step forward using target's argmax (greedy decode)
        next_token = torch.tensor([[target_argmax]], device="cuda")
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if target_argmax == tokenizer.eos_token_id:
            break

    return match_ranks


# Run measurement
all_match_ranks = []
per_prompt_ranks = {}

print("Measuring match ranks across prompts...")
for pidx, prompt in enumerate(PROMPTS):
    label = PROMPT_LABELS[pidx]
    t0 = time.time()
    ranks = measure_match_ranks(prompt)
    elapsed = time.time() - t0
    all_match_ranks.extend(ranks)
    per_prompt_ranks[label] = ranks

    # Per-prompt quick stats
    rank1 = sum(1 for r in ranks if r == 1) / len(ranks)
    any_match = sum(1 for r in ranks if r is not None) / len(ranks)
    print(f"  {label:<16} {len(ranks):>3} positions  p(top-1 match)={rank1:.3f}  p(any top-{K_MAX} match)={any_match:.3f}  ({elapsed:.1f}s)")

print(f"\nTotal samples: {len(all_match_ranks)}")

Measuring match ranks across prompts...
  general_1         32 positions  p(top-1 match)=0.875  p(any top-5 match)=0.969  (3.5s)
  general_2         32 positions  p(top-1 match)=0.875  p(any top-5 match)=1.000  (2.2s)
  general_3         32 positions  p(top-1 match)=0.844  p(any top-5 match)=1.000  (2.2s)
  code_1            17 positions  p(top-1 match)=1.000  p(any top-5 match)=1.000  (1.2s)
  code_2            32 positions  p(top-1 match)=0.781  p(any top-5 match)=0.938  (2.2s)
  summarization     32 positions  p(top-1 match)=0.875  p(any top-5 match)=0.938  (2.7s)
  instruction_1     32 positions  p(top-1 match)=0.656  p(any top-5 match)=1.000  (2.2s)
  instruction_2     32 positions  p(top-1 match)=0.875  p(any top-5 match)=1.000  (2.2s)
  long_ctx_1        32 positions  p(top-1 match)=0.875  p(any top-5 match)=1.000  (2.5s)
  long_ctx_2        32 positions  p(top-1 match)=0.875  p(any top-5 match)=1.000  (2.5s)

Total samples: 305


In [7]:
# ============================================================
# Compute acceptance vector p[1..K_MAX]
# ============================================================
#
# Definitions (positional acceptance, Sequoia Def 3.1):
#   p_b = P(the b-th draft candidate matches target's argmax,
#          given that candidates 1..b-1 did not match)
#
# Equivalently: p_b = count(rank == b) / count(rank >= b)

N = len(all_match_ranks)
p_vector = []
support = []  # number of samples contributing to each p_b (for CI / std-err)

for b in range(1, K_MAX + 1):
    # Count positions where rank == b and where rank >= b (or rank is None, i.e. rank > K_MAX)
    eligible = sum(1 for r in all_match_ranks if (r is None) or (r >= b))
    accepted = sum(1 for r in all_match_ranks if r == b)
    p_b = accepted / eligible if eligible > 0 else 0.0
    p_vector.append(p_b)
    support.append(eligible)

# Also compute absolute mass: P(rank == b) over all samples — useful as a sanity check
mass = [sum(1 for r in all_match_ranks if r == b) / N for b in range(1, K_MAX + 1)]
no_match = sum(1 for r in all_match_ranks if r is None) / N

print("Measured acceptance vector (for Sequoia DP):")
print(f"{'b':>3} {'p_b (cond.)':>13} {'support':>9} {'P(rank==b)':>12}")
for b in range(1, K_MAX + 1):
    print(f"{b:>3} {p_vector[b-1]:>13.4f} {support[b-1]:>9} {mass[b-1]:>12.4f}")
print(f"{'--':>3} {'':>13} {'':>9} {'':>12}")
print(f"Total samples: {N}")
print(f"P(no match in top-{K_MAX}): {no_match:.4f}")

# Sanity: cumulative should equal 1 - no_match
cum = sum(mass)
print(f"Σ P(rank==b) + P(no match) = {cum + no_match:.4f}  (should be 1.0)")

Measured acceptance vector (for Sequoia DP):
  b   p_b (cond.)   support   P(rank==b)
  1        0.8459       305       0.8459
  2        0.5745        47       0.0885
  3        0.5500        20       0.0361
  4        0.3333         9       0.0098
  5        0.1667         6       0.0033
 --                                     
Total samples: 305
P(no match in top-5): 0.0164
Σ P(rank==b) + P(no match) = 1.0000  (should be 1.0)


In [8]:
# ============================================================
# Per-category acceptance (for the report — shows domain effect)
# ============================================================
from collections import defaultdict

category_of = {
    "general_1": "general", "general_2": "general", "general_3": "general",
    "code_1": "code", "code_2": "code",
    "summarization": "summarization",
    "instruction_1": "instruction", "instruction_2": "instruction",
    "long_ctx_1": "long_ctx", "long_ctx_2": "long_ctx",
}

cat_ranks = defaultdict(list)
for label, ranks in per_prompt_ranks.items():
    cat = category_of[label]
    cat_ranks[cat].extend(ranks)

print(f"{'Category':<15} {'p_1':>6} {'p_2':>6} {'p_3':>6} {'no_match':>9} {'n':>5}")
print("-" * 55)
for cat, ranks in cat_ranks.items():
    n = len(ranks)
    def pb(b):
        elig = sum(1 for r in ranks if (r is None) or (r >= b))
        acc  = sum(1 for r in ranks if r == b)
        return acc / elig if elig > 0 else 0.0
    nm = sum(1 for r in ranks if r is None) / n
    print(f"{cat:<15} {pb(1):>6.3f} {pb(2):>6.3f} {pb(3):>6.3f} {nm:>9.3f} {n:>5}")

Category           p_1    p_2    p_3  no_match     n
-------------------------------------------------------
general          0.865  0.538  0.667     0.010    96
code             0.857  0.143  0.333     0.041    49
summarization    0.875  0.500  0.000     0.062    32
instruction      0.766  0.600  0.833     0.000    64
long_ctx         0.875  1.000  0.000     0.000    64


In [9]:
# ============================================================
# Save for Phase 2B
# ============================================================
output = {
    "target_model": TARGET_MODEL,
    "draft_model": DRAFT_MODEL,
    "k_max": K_MAX,
    "n_samples": N,
    "acceptance_vector": p_vector,
    "p_rank_b": mass,
    "p_no_match": no_match,
    "support_per_position": support,
    "per_prompt_ranks": {k: [r if r is not None else -1 for r in v] for k, v in per_prompt_ranks.items()},
}

with open("swor_acceptance_vector.json", "w") as f:
    json.dump(output, f, indent=2)

print("=" * 60)
print("SAVED: swor_acceptance_vector.json")
print("=" * 60)
print(f"Acceptance vector p[1..{K_MAX}] = {[round(x, 4) for x in p_vector]}")
print()
print("This vector is the input to Phase 2B's Sequoia DP.")
print("Download this file and upload it to the Phase 2B notebook.")

SAVED: swor_acceptance_vector.json
Acceptance vector p[1..5] = [0.8459, 0.5745, 0.55, 0.3333, 0.1667]

This vector is the input to Phase 2B's Sequoia DP.
Download this file and upload it to the Phase 2B notebook.
